# two-optimizers-alternating-step — ex1: alternating D-step then G-step with two optimizers on toy modules

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `two-optimizers-alternating-step`. Running the final beacon cell reports progress against the `GAN: Two-optimizers alternating step` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `GAN: Two-optimizers alternating step` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`two-optimizers-alternating-step`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "two-optimizers-alternating-step"
DD_SUBTOPIC = "GAN: Two-optimizers alternating step"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## GAN: two-optimizers alternating step — quick refresher

GAN training is two ADVERSARIAL optimization problems wired into one loop. Each iteration takes a D-step then a G-step (or vice versa). The canonical pattern:

```python
for x_real in dataloader:
    # === D-step ===
    D_opt.zero_grad()
    z = torch.randn(B, z_dim)
    fake = G(z).detach()              # stop-gradient into G
    loss_D = -(D(x_real).log() + (1 - D(fake)).log()).mean()
    loss_D.backward()
    D_opt.step()

    # === G-step ===
    G_opt.zero_grad()
    z = torch.randn(B, z_dim)
    fake = G(z)                       # GRAD flows into G this time
    loss_G = -D(fake).log().mean()
    loss_G.backward()
    G_opt.step()
```

**Two optimizers, two parameter sets.** `D_opt = Adam(D.parameters(), ...)` only knows about D. `G_opt = Adam(G.parameters(), ...)` only knows about G. Mixing params across optimizers is the most common GAN bug — `G_opt.step()` would silently change D's weights too.

**Zero the OWNED grads only.** `D_opt.zero_grad()` clears grads on D's params but leaves G's grads alone (G doesn't have any yet anyway). Calling `model.zero_grad()` is also fine if you keep both modules separate.

**Order matters for the loss landscape.** D-then-G is the original Goodfellow recipe. G-then-D is also common (WGAN-GP). Whatever the order, both steps happen in each iteration.

### Exercise 1 — alternating D-step then G-step with two optimizers on toy modules

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the GAN alternating-step pattern — D-step then G-step, each with its OWN optimizer and zero_grad — on toy modules to verify only the right module moves per step.
> Keywords: gan, two-optimizers, alternating, d-step, g-step
> ```

**KCs targeted:** `d-step-vs-g-step-ordering`, `per-module-optimizer-isolation`

Implement `ex1_gan_iter(G, D, G_opt, D_opt, z, x_real)`. ONE iteration of the canonical GAN training loop.

STEP 1 — D-step (train the discriminator):
  a. `D_opt.zero_grad()`
  b. Compute `fake = G(z).detach()` (stop-gradient — see the `detach-stop-gradient-trick` drill for why).
  c. Compute `loss_D = (D(fake) - D(x_real)).mean()` (simplified Wasserstein-style loss — we want D to assign high values to reals and low values to fakes, so the loss is fake-real and we MINIMIZE).
  d. `loss_D.backward()`
  e. `D_opt.step()`

STEP 2 — G-step (train the generator):
  a. `G_opt.zero_grad()`
  b. Compute `fake = G(z)` (NO detach — gradient flows through D and into G).
  c. Compute `loss_G = -D(fake).mean()` (G wants D to assign HIGH values to fakes, so we minimize negative D).
  d. `loss_G.backward()`
  e. `G_opt.step()`

Return `(loss_D_value, loss_G_value)` as Python floats.

Inputs:
- `G`, `D`: `nn.Module`s.
- `G_opt`, `D_opt`: optimizers, each constructed with the matching module's parameters.
- `z`: noise input to G, shape `(B, z_dim)`.
- `x_real`: real samples, shape `(B, x_dim)`.

Output: `(float, float)` — D-loss then G-loss values.

In [ ]:
def ex1_gan_iter(G, D, G_opt, D_opt, z, x_real):
    # === D-step ===
    D_opt.zero_grad()
    fake = G(z).detach()                 # stop-gradient into G
    loss_D = (D(fake) - D(x_real)).mean()
    loss_D.backward()
    D_opt.step()

    # === G-step ===
    G_opt.zero_grad()
    fake = G(z)                          # grad flows into G
    loss_G = -D(fake).mean()
    loss_G.backward()
    G_opt.step()

    return loss_D.item(), loss_G.item()


<details><summary>Solution</summary>

```python
def ex1_gan_iter(G, D, G_opt, D_opt, z, x_real):
    # === D-step ===
    D_opt.zero_grad()
    fake = G(z).detach()                 # stop-gradient into G
    loss_D = (D(fake) - D(x_real)).mean()
    loss_D.backward()
    D_opt.step()

    # === G-step ===
    G_opt.zero_grad()
    fake = G(z)                          # grad flows into G
    loss_G = -D(fake).mean()
    loss_G.backward()
    G_opt.step()

    return loss_D.item(), loss_G.item()
```

**Why two `zero_grad` calls.** Each optimizer owns its own parameters' `.grad`. `D_opt.zero_grad()` only clears D's grads; G's grads accumulate untouched. If you called `G_opt.zero_grad()` at the top of the D-step it would do nothing wrong, but the convention is to zero each optimizer right before you use it.

**Why we call `G(z)` TWICE.** Once with `.detach()` during the D-step (D needs to see fakes but G shouldn't learn from D's loss), and again without detach during the G-step (G learns from D's gradient flowing back). The two forward passes are NOT redundant — they build different autograd graphs.

**The cross-wiring guard is the #1 GAN bug.** `G_opt = Adam(model.parameters())` (where `model` accidentally contains both G and D) is a silent killer: every G-step also moves D, and D's training is corrupted in a way that's invisible without an explicit param-set check.

**Order: D-then-G is Goodfellow.** Some papers (WGAN-GP) do multiple D-steps per G-step for better D convergence. The skeleton is identical — just put the D-step in a loop.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()